# Multimodal RAG 실습
PDF → 텍스트/표/이미지 분리 → 요약 → 다중 벡터 검색기 → 최종 체인

대상: `data/sample_text.pdf` (현대자동차 2026년 3분기 실적)

## 구조 요약
1. **추출** — `unstructured hi_res`로 텍스트·표·이미지를 분리(그래프 수치는 이미지 안에 있어 텍스트만으로는 접근 불가).
2. **요약·색인** — 각 요소를 요약해 Chroma에 넣고, 원본은 같은 `doc_id`로 docstore에 보관. 검색은 요약으로, 답변 생성은 원본(이미지 포함)으로 합니다.
3. **답변** — 검색된 원본에서 base64 이미지와 텍스트를 분리해 `gpt-4o-mini`(비전)에 함께 전달.

## 기술 선택 근거
- `gpt-4o-mini`: 비전 포함 + 저렴(`gpt-4-vision-preview`는 지원 종료)
- `hi_res` + `languages=["kor","eng"]`: 표·이미지 추출 정확도, 한글 OCR
- 청크 2000토큰/겹침 200: 경계에서 문장이 끊겨 뜻을 잃는 것을 방지
- 이미지 요약 프롬프트는 영어 지시 + 한국어 출력: 한국어 질문과 검색을 맞추기 위함

In [1]:
import sys, os, time, re, pickle
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent            # notebooks/ 의 상위 = 프로젝트 루트
sys.path.insert(0, str(ROOT))
load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OpenAI API 키가 없습니다. .env 를 확인하세요.")
print("키 로드 OK (값은 출력하지 않음)")

키 로드 OK (값은 출력하지 않음)


## 1. 데이터 추출 및 분할
`hi_res` + 한국어 OCR. 2~3분 걸립니다.

In [2]:
from src.extract import extract_pdf_elements, categorize_elements, split_texts

fname, fpath = "sample_text.pdf", ROOT / "data"
figures = fpath / "figures"
figures.mkdir(exist_ok=True)

raw_pdf_elements = extract_pdf_elements(fpath, fname, figures)
texts, tables = categorize_elements(raw_pdf_elements)
texts_2k_token = split_texts(texts)

jpgs = sorted(figures.glob("*.jpg"))
print("텍스트 블록:", len(texts), "/ 분할 후:", len(texts_2k_token), "/ 표:", len(tables))
print("figures/ jpg:", len(jpgs), "개")
assert jpgs and texts and texts_2k_token

Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Loading weights:  51%|█████     | 186/367 [00:00<00:00, 1848.00it/s]

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 2412.79it/s]

텍스트 블록: 10 / 분할 후: 2 / 표: 10
figures/ jpg: 25 개


In [3]:
print(texts[0][:400])

@P HYUNDAI

YW WY LLL % Wi Wh

현 대 자동차 20264 3 분 기 SSA 발표 2026 [도 =" jo) [20 NO 0) me

유 의 사항

본 자 료 는 2026 년 3 분 기 실 적 에 대한 외 부 감 사 인의 회 계 감 사 가 완 료 되지 않은 상 태 에서 투자자 여 러 분 들 의 편 의 를 위하여 작성된 자 료 로서, 제시된 자 료 의 내용 중 일 부 는 회 계 감사 과 정 에서 ASS 수 있 음 을 양 지 하시기 바랍니다.


## 2. 텍스트·표 요약

In [4]:
from src.summarize import generate_text_summaries

text_summaries, table_summaries = generate_text_summaries(texts_2k_token, tables, summarize_texts=True)
print(len(text_summaries), len(table_summaries))
print(text_summaries[0])

2 10
**현대자동차 2026년 3분기 실적 요약**

- **판매 성과**: 
  - 미국 시장에서 소매 판매가 전년 대비 3.5% 증가하며 시장 점유율(M/S) 6.6% 달성.
  - 글로벌 HEV(하이브리드 전기차) 판매 비중 18.2%로, 분기 최대 비중인 26.8% 기록.

- **산업 수요 변화**: 
  - 2025년 대비 2026년 3분기 도매 판매량은 -11.7% 감소, 소매 판매량은 +3.5% 증가.
  - 지역별로는 북미, 유럽, 아시아태평양에서 각각 -18.0%, -4.5%, -14.5% 감소.

- **차급별 판매**: 
  - SUV와 PV(승용차) 판매 비중은 각각 22.1%와 77.9%로 나타남.
  - HEV와 PHEV 판매 비중은 각각 6.8%와 5.1%로 증가.

- **기타 사항**: 
  - 본 자료는 외부 감사 완료 전의 정보로, 일부 내용은 변경될 수 있음. 

이 요약은 현대자동차의 2026년 3분기 실적과 관련된 주요 지표 및 변화를 간결하게 정리한 것입니다.


## 3. 이미지 요약

In [5]:
from src.summarize import generate_img_summaries

img_base64_list, image_summaries = generate_img_summaries(str(figures))
print(len(image_summaries), "장 요약")
print("base64 앞 100자:", img_base64_list[0][:100])
print(image_summaries[0])

25 장 요약
base64 앞 100자: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAx
현대 SUV 차량이 현대적인 건축 구조물 앞에 주차되어 있는 모습. 차량은 미래지향적인 디자인이며, 배경은 금속성과 기하학적 형태가 특징인 건물이다.


## 4. 다중 벡터 검색기

In [6]:
from src.retriever import create_retriever, add_documents, build_chain

retriever = create_retriever()
add_documents(retriever, text_summaries, texts_2k_token)
add_documents(retriever, table_summaries, tables)
add_documents(retriever, image_summaries, img_base64_list)
chain = build_chain(retriever)
print("색인 완료")

색인 완료


## 5. 최종 체인 실행

In [7]:
query = "그래프에 나타난 추세를 설명해줘"
t = time.time()
print(chain.invoke(query))
print(f"\n소요 {time.time()-t:.1f}초")

그래프에 나타난 추세는 다음과 같습니다:

1. **첫 번째 그래프**에서는 전반적으로 감소세가 보입니다. 특히, 두 번째 바에서 -9.9%의 감소가 나타나고 있으며, 이후 세 번째와 네 번째 바에서는 소폭의 증가(+3.5%)가 있습니다.

2. **두 번째 그래프**에서는 전반적으로 증가세가 뚜렷합니다. 특히, 두 번째 바에서 +18.8%의 증가가 있으며, 나머지 바들도 모두 증가세를 보이고 있습니다.

3. **세 번째 그래프**는 원형 그래프로, 각 부분의 비율이 표시되어 있습니다. 가장 큰 비율인 62.0%가 차지하고 있으며, 전체적으로 긍정적인 추세를 나타냅니다.

4. **네 번째 그래프**에서는 두 개의 원형 그래프가 비교되고 있습니다. 첫 번째 그래프는 69의 수치를 가지고 있으며, 두 번째 그래프는 57의 수치를 가지고 있습니다. 두 그래프 모두 특정 비율을 나타내고 있으며, 첫 번째 그래프가 더 높은 수치를 보이고 있습니다.

전반적으로 첫 번째 그래프는 감소세, 두 번째 그래프는 증가세를 보이며, 세 번째와 네 번째 그래프는 비율을 통해 긍정적인 추세를 나타내고 있습니다.

소요 4.8초


## 6. Before / After (텍스트 RAG vs 멀티모달 RAG)

In [8]:
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vs = Chroma(collection_name="text_only_rag", embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"))
vs.add_texts(texts_2k_token)
prompt = ChatPromptTemplate.from_template(
    "아래 자료를 근거로 질문에 한국어로 답하세요.\n자료에 없는 내용은 모른다고 답하세요.\n\n질문: {question}\n\n참고 텍스트:\n{context}")
join = lambda docs: "\n".join(d.page_content for d in docs)
text_chain = ({"context": vs.as_retriever(search_kwargs={"k": 4}) | join, "question": RunnablePassthrough()}
              | prompt | ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024) | StrOutputParser())

print("[BEFORE] 텍스트 전용\n", text_chain.invoke(query))
print("\n[AFTER] 멀티모달\n", chain.invoke(query))

[BEFORE] 텍스트 전용
 자료에 따르면, 그래프에서 나타난 추세는 HEV(하이브리드 전기차) 판매가 증가하고 있으며, 특히 도매 판매 기준으로 HEV의 비중이 26.8%로 역대 최대치를 기록했다는 점입니다. 또한, 소매 판매에서도 HEV의 비중이 6.6%에 달하며, 6개 분기 연속으로 6% 대를 유지하고 있습니다. 반면, 유럽 시장에서는 소형 ICE(내연기관) 라인업의 효율화와 xEV(전기차) 중심의 라인업 전환으로 인해 단기적인 판매에 부정적인 영향을 미치고 있습니다. 전반적으로 HEV와 xEV의 판매 비중이 증가하는 추세가 보입니다.



[AFTER] 멀티모달
 그래프에 나타난 추세는 다음과 같습니다:

1. **첫 번째 그래프**에서는 전반적으로 감소세가 보입니다. 특히, 두 번째 바에서 -9.9%의 감소가 나타나고 있으며, 이후 세 번째와 네 번째 바에서는 각각 +3.5%의 증가가 있습니다. 이는 초기 감소 후 점진적인 회복을 나타냅니다.

2. **두 번째 그래프**에서는 전반적으로 증가세가 뚜렷합니다. 특히, 두 번째 바에서 +18.8%의 증가가 있으며, 전체적으로 긍정적인 성장 추세를 보여줍니다.

3. **세 번째 그래프**는 비율을 나타내며, 921이라는 수치가 중심에 위치하고 있습니다. 각 섹션의 비율이 다르게 나타나며, 특정 부분이 전체에서 차지하는 비율이 강조되고 있습니다.

4. **네 번째 그래프**에서는 두 개의 원형 그래프가 비교되고 있으며, 각각의 비율이 다르게 나타납니다. 첫 번째 그래프는 69라는 수치를 중심으로 하고, 두 번째 그래프는 57을 중심으로 하고 있습니다. 두 그래프 모두 특정 비율의 증가와 감소를 보여줍니다.

전반적으로, 첫 번째 그래프는 감소 후 회복을, 두 번째 그래프는 지속적인 성장을, 세 번째와 네 번째 그래프는 비율의 분포를 나타내고 있습니다.


## 7. 질의응답 5건

In [9]:
IMG_RE = re.compile(r"^[A-Za-z0-9+/]{100,}={0,2}$")
def kinds(docs):
    k = ["이미지" if IMG_RE.match(d) else "표" if d in tables else "텍스트" for d in docs]
    return ", ".join(f"{x} {k.count(x)}" for x in dict.fromkeys(k))

retriever.search_kwargs = {"k": 8}  # 질의응답에서만 검색 후보를 4 -> 8장으로 늘림
questions = [q.strip() for q in (ROOT / "data/questions.txt").read_text(encoding="utf-8").splitlines() if q.strip()]
rows = ["| # | 질문 | 답변 | 근거 원본 종류 | 소요(초) |", "|---|---|---|---|---|"]
for i, q in enumerate(questions, 1):
    t = time.time()
    a = chain.invoke(q)
    rows.append(f"| {i} | {q} | {a.strip().replace(chr(10), '<br>').replace('|', '/')} | {kinds(retriever.invoke(q))} | {time.time()-t:.1f} |")
table_md = "\n".join(rows)
(ROOT / "qa_table.md").write_text(table_md, encoding="utf-8")
from IPython.display import Markdown
Markdown(table_md)

| # | 질문 | 답변 | 근거 원본 종류 | 소요(초) |
|---|---|---|---|---|
| 1 | 파워트레인별 판매 표에서 HEV 도매 판매량은 Q3 2025와 Q3 2026에 각각 몇 천 대인가? | HEV 도매 판매량은 다음과 같습니다:<br><br>- Q3 2025: 161천 대<br>- Q3 2026: 167천 대 | 이미지 4, 텍스트 2, 표 2 | 5.7 |
| 2 | 인도 시장의 산업수요·도매·소매 판매량 증감률을 그래프에서 알려줘 | 인도 시장의 산업수요, 도매, 소매 판매량 증감률은 다음과 같습니다:<br><br>- **산업수요**: +30.1%<br>- **도매**: +18.8%<br>- **소매**: +9.5% | 이미지 8 | 5.0 |
| 3 | 중국 시장의 도매 판매량은 Q3 2025 대비 Q3 2026에 몇 % 변했나? | 자료에 따르면, 중국 시장의 도매 판매량은 Q3 2025 대비 Q3 2026에 약 -9.4% 변했습니다. Q3 2025의 도매 판매량은 1,006천 대였고, Q3 2026의 도매 판매량은 911천 대로 나타났습니다. | 이미지 5, 표 2, 텍스트 1 | 4.8 |
| 4 | 영업이익은 Q3 2025 대비 Q3 2026에 얼마나 변했나? 표의 단위를 확인해서 답해줘 | 영업이익은 Q3 2025에서 Q3 2026으로 변동이 있었습니다. Q3 2025의 영업이익은 2,537억 원이었고, Q3 2026의 영업이익은 2,677억 원입니다. 따라서 영업이익은 140억 원 증가했습니다. | 이미지 5, 표 3 | 6.0 |
| 5 | 글로벌 도매 판매량은 Q3 2025 대비 Q3 2026에 몇 % 변했나? | 자료에 따르면, Q3 2025 대비 Q3 2026의 글로벌 도매 판매량은 -9.9% 변했습니다. | 이미지 3, 표 4, 텍스트 1 | 3.8 |

## Before / After 해설
Before(텍스트 전용)는 이미지가 색인에 없어서 본문에 적힌 문장만 근거로 답합니다. After(멀티모달)는 이미지 요약으로 원본 그래프를 찾아 모델에 직접 보여 주기 때문에 그래프 속 숫자를 답에 쓸 수 있습니다.

다만 질문이 막연하면 모델이 임의의 그래프를 고릅니다. 위 질의응답 5건(검색 후보 8장)을 원본 슬라이드와 대조하면 정답 2건(1, 2번), 부분 정답 1건(4번: 단위를 십억 원이 아닌 억 원으로 읽음), 오답 2건(3번: 중국 제외 값을 고름, 5번: 다른 지역 그래프 값을 고름)입니다. 후보를 4장으로 했을 때는 3번이 "정보 없음"이었습니다. 정답과 원인은 README 7~8절에 정리했습니다.